
# Analyze Graph

This notebook extends `analyze_episode.ipynb` to generate multiple graphs at once.

It uses the following data sources:

- `replay_efe_checkpoint_{ckpt}_{recorded_episode}.csv`
- `hand_sticker_distance_{recorded_episode}.csv`
- `eval_episode_summary_{checkpoint}.csv`

Generated plots:

- Per-step EFE before/after detach across all seeds and eval episodes (mean bars + std, with per-episode connection lines)
- Total EFE mean by checkpoint across all seeds + 5%~95% band
- Average hand-sticker distance mean by checkpoint across all seeds + 5%~95% band
- Detached-rate mean by checkpoint across all seeds + 5%~95% band


In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
from scipy.stats import binomtest


def format_axis_kM(x, pos):
    """Format tick as 1K, 1M for x-axis."""
    if x >= 1e6:
        return f"{x / 1e6:.0f}M"
    if x >= 1e3:
        return f"{x / 1e3:.0f}K"
    return f"{x:.0f}"


def format_axis_y_k(x, pos):
    """Format y-axis as 10k, 12k (lowercase k for thousands)."""
    if abs(x) >= 1e6:
        return f"{x / 1e6:g}M"
    if abs(x) >= 1e3:
        return f"{x / 1e3:g}k"
    return f"{x:g}"


plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
    }
)


In [ ]:
LOG_ROOT = Path(
    ".../log/"
    "robot_mirror/rep5easy/eval_period500/nodecay"
)

RUN_DIR_NAMES = ["0-0", "1-0", "2-0", "3-0", "4-0", "5-0", "6-0", "7-0"]
# RUN_DIR_NAMES = ["0-0", "1-0"]
CHECKPOINT_EPISODES = list(range(0, 50000 + 1, 500))
# CHECKPOINT_EPISODES = list(range(0, 50000 + 1, 1000))
# CHECKPOINT_EPISODES = list(range(0, 50000 + 1, 5000))
# CHECKPOINT_EPISODES = [0, 10000, 20000, 30000, 40000, 50000]
# CHECKPOINT_EPISODES = [0, 20000, 50000]

RECORDED_EPISODE = 50000
N_EVAL_EPISODES = 10
MAX_PLOT_STEPS = None

SINGLE_RUN_DIR_NAME = "0-0"
SINGLE_CHECKPOINT_EPISODE = 50000

FIGURE_ASPECT = "4:3"
FIGURE_WIDTH_IN = 3
_fig_h = FIGURE_WIDTH_IN  # 1:1
# _fig_h = FIGURE_WIDTH_IN * 3/4  # 4:3
# _fig_h = FIGURE_WIDTH_IN * 9/16  # 16:9
# _fig_h = FIGURE_WIDTH_IN * 16/9  # 9:16

CONFIDENCE_LEVEL = 0.95
BINARY_METRIC_NAMES = {"sticker_detached"}

OUTPUT_DIR = LOG_ROOT / "graph_plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def get_run_dir(run_dir_name):
    return LOG_ROOT / run_dir_name

def load_replay_efe_df(
    run_dir_name, checkpoint_episode, recorded_episode, episode_idx=0
):
    csv_path = get_run_dir(run_dir_name) / (
        f"replay_efe_checkpoint_{checkpoint_episode}_{recorded_episode}.csv"
    )
    if csv_path.is_file():
        df = pd.read_csv(csv_path)
        if "episode_idx" in df.columns:
            df = df[df["episode_idx"].astype(int) == int(episode_idx)].copy()
            if len(df) > 0:
                return df

    legacy_csv_path = get_run_dir(run_dir_name) / (
        f"replay_episode_{episode_idx}_efe_checkpoint_{checkpoint_episode}_{recorded_episode}.csv"
    )
    if legacy_csv_path.is_file():
        return pd.read_csv(legacy_csv_path)

    print(f"Missing replay CSV: {csv_path} (and legacy {legacy_csv_path})")
    return None


def load_hand_sticker_distance_df(run_dir_name, recorded_episode, episode_idx=0):
    csv_path = get_run_dir(run_dir_name) / f"hand_sticker_distance_{recorded_episode}.csv"
    if csv_path.is_file():
        df = pd.read_csv(csv_path)
        if "episode_idx" in df.columns:
            df = df[df["episode_idx"].astype(int) == int(episode_idx)].copy()
            if len(df) > 0:
                return df

    legacy_csv_path = get_run_dir(run_dir_name) / (
        f"eval_episode_{episode_idx}_hand_sticker_distance_{recorded_episode}.csv"
    )
    if legacy_csv_path.is_file():
        return pd.read_csv(legacy_csv_path)

    print(f"Missing hand-sticker CSV: {csv_path} (and legacy {legacy_csv_path})")
    return None


def save_current_figure(stem):
    png_path = OUTPUT_DIR / f"{stem}.png"
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    print(f"{png_path}")
    print(f"{pdf_path}")

def plot_band(ax, x_values, center_values, low_values, high_values, label=None, color=None):
    line, = ax.plot(x_values, center_values, label=label, linewidth=2.0, color=color)
    band_color = color if color is not None else line.get_color()
    ax.fill_between(x_values, low_values, high_values, alpha=0.2, color=band_color)
    return line

def collect_replay_before_after_detach_stats(
    run_dir_names,
    checkpoint_episode,
    recorded_episode,
    n_eval_episodes=10,
    max_plot_steps=None,
):
    records = []

    for run_dir_name in run_dir_names:
        for episode_idx in range(n_eval_episodes):
            efe_df = load_replay_efe_df(
                run_dir_name,
                checkpoint_episode,
                recorded_episode,
                episode_idx=episode_idx,
            )
            det_df = load_hand_sticker_distance_df(
                run_dir_name,
                recorded_episode,
                episode_idx=episode_idx,
            )
            if efe_df is None or det_df is None:
                continue
            if "efe" not in efe_df.columns or "is_sticker_detached" not in det_df.columns:
                continue

            efe_values = efe_df["efe"].to_numpy(dtype=float)
            detached_flags = det_df["is_sticker_detached"].to_numpy(dtype=float) == 1

            n_steps = min(len(efe_values), len(detached_flags))
            if max_plot_steps is not None:
                n_steps = min(n_steps, max_plot_steps)
            if n_steps <= 1:
                continue

            efe_values = efe_values[:n_steps]
            detached_flags = detached_flags[:n_steps]
            if not np.any(detached_flags):
                continue

            first_det_step = int(np.argmax(detached_flags))
            before_values = efe_values[:first_det_step]
            after_values = efe_values[first_det_step:]
            if before_values.size == 0 or after_values.size == 0:
                continue

            records.append(
                {
                    "run_dir_name": run_dir_name,
                    "episode_idx": episode_idx,
                    "first_det_step": first_det_step,
                    "before_mean_efe": float(np.mean(before_values)),
                    "after_mean_efe": float(np.mean(after_values)),
                    "before_n_steps": int(before_values.size),
                    "after_n_steps": int(after_values.size),
                }
            )

    return pd.DataFrame.from_records(records)

def build_checkpoint_series_across_runs_mean_band(run_dir_names, checkpoints, metric_name):
    """Compute run-level mean and bands. Uses Wilson CI for binary metrics and quantile bands for continuous metrics."""
    x_values = []
    centers = []
    lows = []
    highs = []

    for checkpoint_episode in checkpoints:
        accumulated = []
        for run_dir_name in run_dir_names:
            csv_path = get_run_dir(run_dir_name) / f"eval_episode_summary_{checkpoint_episode}.csv"
            df = pd.read_csv(csv_path)
            if df is None or metric_name not in df.columns:
                continue
            v = df[metric_name].to_numpy(dtype=float)
            v = v[np.isfinite(v)]
            if v.size > 0:
                accumulated.extend(v)

        accumulated = np.asarray(accumulated, dtype=float)
        x_values.append(checkpoint_episode)
        center = float(np.mean(accumulated))
        centers.append(center)

        if metric_name in BINARY_METRIC_NAMES:
            n = int(accumulated.size)
            k = int(np.sum(accumulated > 0.5))
            ci = binomtest(k=k, n=n).proportion_ci(
                confidence_level=CONFIDENCE_LEVEL,
                method="wilson",
            )
            lows.append(float(np.clip(ci.low, 0.0, 1.0)))
            highs.append(float(np.clip(ci.high, 0.0, 1.0)))
        else:
            lows.append(float(np.quantile(accumulated, CONFIDENCE_LEVEL)))
            highs.append(float(np.quantile(accumulated, 1.0 - CONFIDENCE_LEVEL)))

    return (
        np.asarray(x_values, dtype=float),
        np.asarray(centers, dtype=float),
        np.asarray(lows, dtype=float),
        np.asarray(highs, dtype=float),
    )



In [ ]:
# Compare per-step EFE before/after detach: mean bars + std + per-episode connection lines
before_after_df = collect_replay_before_after_detach_stats(
    RUN_DIR_NAMES,
    SINGLE_CHECKPOINT_EPISODE,
    RECORDED_EPISODE,
    n_eval_episodes=N_EVAL_EPISODES,
    max_plot_steps=MAX_PLOT_STEPS,
)
debug_csv = OUTPUT_DIR / "before_after_detach_debug.csv"
before_after_df.to_csv(debug_csv, index=False)
print(f"\nSaved full CSV: {debug_csv}")

# --- No-detach cases: (run, episode) where hand_sticker never becomes detached ---
no_detach_list = []
for run_dir_name in RUN_DIR_NAMES:
    for episode_idx in range(N_EVAL_EPISODES):
        det_df = load_hand_sticker_distance_df(run_dir_name, RECORDED_EPISODE, episode_idx)
        if det_df is None:
            continue
        col = "is_sticker_detached" if "is_sticker_detached" in det_df.columns else ("detached" if "detached" in det_df.columns else None)
        if col is None:
            continue
        if col == "is_sticker_detached":
            has_detach = (det_df[col].to_numpy(dtype=float) == 1).any()
        else:
            has_detach = (det_df[col].astype(bool)).any()
        if not has_detach:
            no_detach_list.append((run_dir_name, episode_idx))

print(f"Episodes that failed to detach sticker: {len(no_detach_list)}")
print(no_detach_list)

# --- Wilcoxon signed-rank (before vs after EFE) ---
from scipy import stats
before_efe = before_after_df["before_mean_efe"].to_numpy(dtype=float)
after_efe = before_after_df["after_mean_efe"].to_numpy(dtype=float)
stat, p_wilcoxon = stats.wilcoxon(before_efe, after_efe)
print("\n--- Before vs After EFE (paired) ---")
print(f"Wilcoxon signed-rank: stat={stat}, p={p_wilcoxon}")
if p_wilcoxon < 0.01:
    print("Interpretation: significant difference (p < 0.01) = **")
elif p_wilcoxon < 0.05:
    print("Interpretation: significant difference (p < 0.05) = *")
else:
    print("Interpretation: no significant difference (p >= 0.05)")

# --- bootstrap CI (before vs after EFE) ---
# BOOTSTRAP_SAMPLES = 2000
# BOOTSTRAP_SEED = 12345
# diff = before_efe - after_efe
# rng = np.random.default_rng(BOOTSTRAP_SEED)
# boot_means = [np.mean(rng.choice(diff, size=len(diff), replace=True)) for _ in range(BOOTSTRAP_SAMPLES)]
# low = np.quantile(boot_means, CONFIDENCE_LEVEL)
# high = np.quantile(boot_means, 1.0 - CONFIDENCE_LEVEL)
# print(f"Mean(before - after) = {np.mean(diff):.4f}, bootstrap {int(CONFIDENCE_LEVEL*100)}% CI = [{low:.4f}, {high:.4f}]")
# if low > 0 or high < 0:
#     print("CI does not include 0 -> significant difference")
# else:
#     print("CI includes 0 -> no significant difference")

before_values = before_after_df["before_mean_efe"].to_numpy(dtype=float)
after_values = before_after_df["after_mean_efe"].to_numpy(dtype=float)

bar_means = np.asarray(
    [np.mean(before_values), np.mean(after_values)],
    dtype=float,
)
bar_stds = np.asarray(
    [
        np.std(before_values, ddof=1) if before_values.size > 1 else 0.0,
        np.std(after_values, ddof=1) if after_values.size > 1 else 0.0,
    ],
    dtype=float,
)

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, _fig_h))
x = np.asarray([0, 1], dtype=float)
bars =ax.bar(
    x,
    bar_means,
    yerr=bar_stds,
    width=0.3,
    capsize=1,
    color=["white", "white"],
    alpha=0.75,
    edgecolor="black",
    linewidth=1.0,
    zorder=2,
)
# Hatch pattern per bar
hatches = [" ", " "]  # before, after
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

rng = np.random.default_rng(12345)
jitter = rng.uniform(-0.05, 0.05, size=len(before_after_df))
for idx in range(len(before_after_df)):
    ax.plot(
        [x[0] + jitter[idx], x[1] + jitter[idx]],
        [before_values[idx], after_values[idx]],
        color="gray",
        alpha=0.25,
        linewidth=1.0,
        zorder=1,
    )
ax.scatter(
    np.full(len(before_after_df), x[0]) + jitter,
    before_values,
    color="white",
    edgecolors="black",
    linewidths=0.5,
    alpha=0.8,
    s=28,
    zorder=3,
)
ax.scatter(
    np.full(len(before_after_df), x[1]) + jitter,
    after_values,
    color="white",
    edgecolors="black",
    linewidths=0.5,
    alpha=0.8,
    s=28,
    zorder=3,
)

# Significance bracket over the two bars (paired Wilcoxon)
def _stars_from_p(p):
    if p >= 0.05:
        return None
    if p < 0.01:
        return "**"
    return "*"


star_txt = _stars_from_p(p_wilcoxon)
if star_txt is not None:
    y_err_top = float(np.max(bar_means + bar_stds))
    y_pts = float(max(np.max(before_values), np.max(after_values)))
    y_base = max(y_err_top, y_pts)
    y_lo, y_hi = ax.get_ylim()
    span = y_hi - y_lo
    pad = max(span * 0.10, 0.02 * max(y_base, 1e-6))
    y_h = y_base + pad
    y1_top = float(bar_means[0] + bar_stds[0] + 5)
    y2_top = float(bar_means[1] + bar_stds[1] + 5)
    y_top = max(y1_top, y2_top)
    ax.plot(
        [x[0], x[0]],
        [y_top, y_h],
        color="black",
        linewidth=1.0,
        clip_on=False,
        zorder=4,
    )
    ax.plot(
        [x[1], x[1]],
        [y_top, y_h],
        color="black",
        linewidth=1.0,
        clip_on=False,
        zorder=4,
    )
    ax.plot(
        [x[0], x[1]],
        [y_h, y_h],
        color="black",
        linewidth=1.0,
        clip_on=False,
        zorder=4,
    )
    ax.text(
        float(np.mean(x)),
        y_h + pad * 0.05,
        star_txt,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        zorder=5,
        clip_on=False,
    )
    new_top = y_h + pad * 1.25
    if y_hi < new_top:
        ax.set_ylim(top=new_top)

ax.set_xticks(x, ["Before detach", "After detach"])
ax.set_xlabel("Condition")
ax.set_ylabel("Expected Free Energy (EFE)")
ax.set_title(f"Average EFE")
ax.grid(True, axis="y", alpha=0.3, zorder=0)
plt.xlim(-0.5, 1.5)
save_current_figure(
    f"replay_efe_before_after_detach_ckpt_{SINGLE_CHECKPOINT_EPISODE}_ep_{RECORDED_EPISODE}"
)
plt.show()

print(bar_means)
print(bar_stds)

In [ ]:
# Mean + 5%~95% band by checkpoint across all seeds (total EFE, average hand-sticker distance, detached rate)

metric_specs = [
    (
        "total_efe",
        "Total Expected Free Energy",
        "summary_total_efe_all_runs",
        None, # "Total EFE in single episode",
    ),
    (
        "avg_hand_sticker_distance",
        "Distance (cm)",
        "summary_avg_distance_all_runs",
        "Mean Hand-Sticker Distance",
    ),
    (
        "sticker_detached",
        "Probability (%)",
        "summary_detached_all_runs",
        "Detached Probability",
    ),
]

for metric_name, y_label, all_stem, title_base in metric_specs:
    key_all = f"summary_{metric_name}_all"

    x_values, centers, lows, highs = build_checkpoint_series_across_runs_mean_band(
        RUN_DIR_NAMES,
        CHECKPOINT_EPISODES,
        metric_name,
    )
    if len(x_values) == 0:
        continue

    # Convert hand-sticker distance from m to cm for display
    if metric_name == "avg_hand_sticker_distance":
        centers = np.asarray(centers, dtype=float) * 100
        lows = np.asarray(lows, dtype=float) * 100
        highs = np.asarray(highs, dtype=float) * 100

    if metric_name == "sticker_detached":
        centers = np.asarray(centers, dtype=float) * 100
        lows = np.asarray(lows, dtype=float) * 100
        highs = np.asarray(highs, dtype=float) * 100

    fig, ax = plt.subplots(
        figsize=(FIGURE_WIDTH_IN, _fig_h),
    )
    episode_steps = x_values
    train_steps = episode_steps * 10
    plot_band(
        ax,
        train_steps,
        centers,
        lows,
        highs,
    )
    ax.set_xlabel("Train steps")
    ax.xaxis.set_major_locator(MultipleLocator(125_000))
    ax.xaxis.set_major_formatter(FuncFormatter(format_axis_kM))
    ax.set_xlim(left=0, right=float(np.max(train_steps)))
    ax.set_ylabel(y_label)
    if metric_name == "total_efe":
        ax.yaxis.set_major_formatter(FuncFormatter(format_axis_y_k))

    if metric_name == "sticker_detached":
        ax.set_ylim(-5, 105)
        
    if title_base is not None:
        ax.set_title(f"{title_base}")
    ax.grid(True)
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend()
    save_current_figure(all_stem)
    plt.show()

